# Diferencias finitas 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_1.ipynb)

### Problemas de valores de contorno: capa límite

Consideremos la ecuación

$$ \varepsilon U_{xx} - U_{x} = f, \qquad U(0)=\alpha, \quad U(1)=\beta. $$

Para el caso $f(x) = 1$, la solución exacta está dada por

$$ U_{\varepsilon}(x) = \alpha + x + (\beta - \alpha -1) \left( \frac{e^{x/\varepsilon}-1 }{e^{1/\varepsilon}-1} \right) $$

> **Nota:** una ecuación de esta forma aparece, por ejemplo, al considerar el estado estacionario ($u_t=0$) de un problema de convección-difusión $u_t = \kappa u_{xx} + a u_x + \phi$, con constantes de difusividad $\kappa>0$ y de convección $a \in \mathbb{R}$. A la proporción $Pe = a/\kappa$ se la conoce como **número de Péclet**, y se toma $\varepsilon = 1/Pe$.

In [ ]:
using LinearAlgebra, Plots

In [ ]:
function U_ε(x,ε;α=1,β=3)
    y=α+x+(β-α-1)*((ℯ^(x/ε)-1)/(ℯ^(1/ε)-1))
    return y
end

Graficamos la solución exacta para $\alpha=1,\ \beta=3$ a medida que $\varepsilon \to 0$:

In [ ]:
x=0:0.01:1

plot(x,U_ε.(x,0.3),label="epsilon=0.3")
plot!(x,U_ε.(x,0.1),label="epsilon=0.1")
plot!(x,U_ε.(x,0.05),label="epsilon=0.05")
plot!(x,U_ε.(x,0.01),label="epsilon=0.01")

A medida que $\varepsilon$ disminuye, la solución desarrolla una transición cada vez más abrupta cerca del borde $x=1$: es la **capa límite**, una región angosta donde la solución cambia muy rápido mientras que en el resto del dominio se mantiene casi constante.

### Resolución numérica

Discretizando con diferencias centradas tanto la derivada primera como la segunda sobre una malla de tamaño $h$, se obtiene un sistema lineal tridiagonal para los valores interiores de $u$:

In [ ]:
function capa_limite(f,N;ε=0.3,α=1,β=3)

    h=1/N
    x=0:h:1
    n=length(x)
    U=zeros(n)
    U[1]=α
    U[n]=β
    A1=(ε/h^2)*Tridiagonal(ones(n-3),-2*ones(n-2),ones(n-3))
    A2=(1/2h)*Tridiagonal(-ones(n-3),zeros(n-2),ones(n-3))
    F=f.(x)[2:n-1]
    F[1]=F[1]-ε*α/h^2+α/2h
    F[end]=F[end]-ε*β/h^2-β/2h

    A=A1-A2
    sol=A\F
    U[2:n-1]=sol
    return U
end

function f(x)
    return -1.0
end

Comparamos la solución numérica con la solución exacta para $\varepsilon = 0.1$ y $N=100$:

In [ ]:
ε=0.1
N=100
x=0:1/N:1
U=capa_limite(f,N,ε=ε)
plot(x,U,label="solucion numerica")
plot!(x,u_ε.(x,ε),label="solucion exacta")

**Conclusión:** la aproximación numérica reproduce bien tanto la zona suave como la capa límite cerca de $x=1$, siempre que la malla sea suficientemente fina en relación a $\varepsilon$. Si $h \gg 2\varepsilon$, el esquema deja de resolver correctamente la capa límite y aparecen oscilaciones espurias — se puede verificar variando `N` y `ε` en la celda anterior.